# TFEX `USDM26` — live bid/ask from the Settrade Open API

Connect to the **Settrade Open API (sandbox)** and read the **bid/ask order book** for `USDM26`
— the TFEX USD futures contract (1 contract = USD 1,000, cash-settled, last trade 29 Jun 2026).
This is the *live-data* companion to Step 1, which used saved 1-minute candles.

> **Sandbox + read-only.** Uses the sandbox environment and only *reads* market data — it never
> places an order. Educational only, not investment advice.

**Two ways to get bid/ask, in order of reliability:**
1. **Snapshot** — `get_quote_symbol` returns the current top-of-book on demand. Use this first.
2. **Live ladder** — `subscribe_bid_offer` streams the full 5-level ladder, but only fires on the
   *next* order-book change, so it needs an active market and a little patience.

## 1. Install (run once)

In [ ]:
# Already in this repo's uv environment. Uncomment if running elsewhere:
# !pip install settrade-v2 python-dotenv pandas

## 2. Imports

In [ ]:
import os
import json
import threading

import pandas as pd
from dotenv import load_dotenv
from settrade_v2 import Investor

## 3. Connect to the sandbox

Credentials come from `.env` (copy `.env.example` → `.env` and fill in your sandbox keys).

> **The one gotcha that wastes an afternoon:** for the **sandbox** the SDK expects
> `app_code="SANDBOX"` **and `broker_id="098"`** (these are `Investor.SANDBOX` and
> `Investor.SANDBOX_BROKER_ID` in the SDK). Passing `broker_id="SANDBOX"` fails login with
> `U-102: Invalid Signature`. The helper below forces the correct values in sandbox, so it works
> even if `.env` still has `SETTRADE_BROKER_ID=SANDBOX`.

In [ ]:
load_dotenv(override=True)

IS_SANDBOX = os.environ.get("SETTRADE_ENV", "sandbox").lower() in ("sandbox", "uat")

def connect():
    """Return a logged-in Investor. Forces the correct sandbox app_code/broker_id."""
    return Investor(
        app_id=os.environ["SETTRADE_APP_ID"],
        app_secret=os.environ["SETTRADE_APP_SECRET"],
        app_code="SANDBOX" if IS_SANDBOX else os.environ["SETTRADE_APP_CODE"],
        broker_id="098"    if IS_SANDBOX else os.environ["SETTRADE_BROKER_ID"],
        is_auto_queue=False,
    )

try:
    investor = connect()          # login happens here
    mkt = investor.MarketData()
    print(f"Connected. sandbox={IS_SANDBOX}")
except Exception as e:
    # Don't let a bad key / blocked host hang silently — surface it.
    raise RuntimeError(
        f"Settrade login failed: {type(e).__name__}: {e}\n"
        "Check SETTRADE_APP_ID/SECRET, and that this network can reach open-api.settrade.com."
    ) from e

## 4. Snapshot: best bid / ask right now

`get_quote_symbol` returns one dict with the latest trade, day range, and top-of-book. We print
the raw response first so you can see every field this contract exposes.

In [ ]:
SYMBOL = "USDM26"

quote = mkt.get_quote_symbol(symbol=SYMBOL)
print(json.dumps(quote, indent=2, default=str))

Read the top-of-book defensively — field names vary slightly between contract types — and compute the spread, which is what a market-neutral trader pays to cross the book.

In [ ]:
def first(d, *keys, default=None):
    """First present, non-None key from a dict."""
    for k in keys:
        if isinstance(d, dict) and d.get(k) is not None:
            return d[k]
    return default

bid  = first(quote, "bid", "bidPrice", "bidPrice1")
ask  = first(quote, "ask", "askPrice", "askPrice1", "offer", "offerPrice")
last = first(quote, "last", "lastPrice", "price")
high = first(quote, "high", "highPrice")
low  = first(quote, "low", "lowPrice")

spread = (ask - bid) if (bid is not None and ask is not None) else None
mid    = ((ask + bid) / 2) if (bid is not None and ask is not None) else None

print(f"{SYMBOL}")
print(f"  best bid : {bid}")
print(f"  best ask : {ask}")
print(f"  spread   : {spread}")
print(f"  mid      : {mid}")
print(f"  last     : {last}   (day high {high} / low {low})")

## 5. Live ladder: the full bid/ask depth

`subscribe_bid_offer` opens a realtime feed and calls your `on_message` callback **every time the
order book changes**. Two things to know, both of which made earlier versions look "stuck":

- **The callback gets a wrapped dict**, not the fields directly:
  `{"is_success": True, "data": {"bid_price1": ..., "bid_volume1": ..., "ask_price1": ..., ...}}`.
  The ladder lives under `msg["data"]`. Reading `msg["bid_price1"]` gives `None`.
- **It fires on the next *change*, not on demand.** On a quiet contract or outside market hours no
  message may arrive — so we wait with a **timeout** and report clearly instead of blocking forever.

We wait for one snapshot, then stop the subscription from the main thread.

In [ ]:
result = {}                 # filled by the callback
got = threading.Event()     # signalled when a message arrives

def on_bid_offer(msg):
    # msg = {'is_success': True, 'data': {...}}  or  {'is_success': False, 'message': '...'}
    if msg.get("is_success"):
        result["data"] = msg["data"]
    else:
        result["error"] = msg.get("message") or msg.get("data")
    got.set()

rt = investor.RealtimeDataConnection()
sub = rt.subscribe_bid_offer(symbol=SYMBOL, on_message=on_bid_offer)
sub.start()

arrived = got.wait(timeout=30)   # block up to 30s for the first order-book update
sub.stop()                       # always stop — frees the connection

if not arrived:
    print("No order-book update within 30s.")
    print("Likely the market is closed or USDM26 is quiet right now. The Step-4 snapshot above")
    print("still shows the last known top-of-book. Try again during TFEX trading hours.")
elif "error" in result:
    print("Subscription error:", result["error"])
else:
    print("Got a live ladder snapshot.")

Lay the ladder out the way a trading screen does — bids on the left, asks on the right. Prices come through as floats and volumes as ints.

In [ ]:
LEVELS = 5
data = result.get("data", {})

rows = [{
    "level": i,
    "bid_volume": data.get(f"bid_volume{i}"),
    "bid_price":  data.get(f"bid_price{i}"),
    "ask_price":  data.get(f"ask_price{i}"),
    "ask_volume": data.get(f"ask_volume{i}"),
} for i in range(1, LEVELS + 1)]

ladder = pd.DataFrame(rows).set_index("level")
# empty depth levels arrive as 0 — blank them per side for a clean trading-screen view
ladder.loc[ladder["bid_price"] == 0, ["bid_price", "bid_volume"]] = pd.NA
ladder.loc[ladder["ask_price"] == 0, ["ask_price", "ask_volume"]] = pd.NA
ladder

## 6. Notes

- **Snapshot vs stream.** `get_quote_symbol` (Step 4) answers "what is the book *now*" on demand;
  `subscribe_bid_offer` (Step 5) pushes every change. For a one-off reading the snapshot is simpler.
- **Market hours.** Outside TFEX hours the stream is silent and snapshot bid/ask may be `None`;
  `last`/`high`/`low` still reflect the previous session.
- **Sandbox prices** are simulated and will differ from the live screen.
- **Read-only.** This notebook never calls `place_order`. See `settrade_open_api_setting.ipynb`
  for the (sandbox) order-placement example.
- **Next step.** Logging this bid/ask to a database over time is exactly what `THE_DATA_LOGGER.md`
  describes — the source of the real bid/ask in Steps 2, 4 and 5.